## Mart_Table (Sales Overview) GOLD LAYER INSERTION

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

### Importing Libraries

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum, avg, count, countDistinct,
    lag, round as spark_round, col, year, month
)
from pyspark.sql import Window

#### What is our revenue, AOV, growth rate, and top performers by month/state/category?

In [0]:
# Load tables
df_fact = spark.table("olist_ecommerce_project.gold.fact_orders")
df_date = spark.table("olist_ecommerce_project.gold.dim_date")
df_order_items = spark.table("olist_ecommerce_project.silver.slv_order_items")
df_products = spark.table("olist_ecommerce_project.gold.dim_products")

# Join: fact → date (for month/year) → order_items → products (for category)
df_sales = (
    df_fact
    .join(df_date, on="date_key", how="inner")
    .join(df_order_items, on="order_id", how="inner")
    .join(df_products, on="product_id", how="left")
)

# Aggregate by month
df_mart_sales = (
    df_sales
    .groupBy("year", "month_number", "month_name")
    .agg(
        spark_sum("total_order_value").alias("gross_revenue"),
        count(col("order_id")).alias("total_orders"),
        countDistinct("customer_id").alias("total_customers"),
        spark_round(avg("total_order_value"), 2).alias("avg_order_value"),
        spark_round(avg("total_freight_value"), 2).alias("avg_freight_value"),
        spark_round(
            spark_sum("total_freight_value") / spark_sum("total_order_value") * 100,
            2
        ).alias("freight_to_revenue_ratio_pct")
    )
    .orderBy("year", "month_number")
)

print("mart_sales_overview rows:", df_mart_sales.count())
df_mart_sales.show(10, truncate=False)



In [0]:
# Write to Gold
(
    df_mart_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.mart_sales_overview")
)

print("mart_sales_overview written successfully")